# Day 42: Gradient Descent from Scratch

## The Idea
Most ML models are "trained" by finding parameter values that minimize a loss function (a measure of how wrong the predictions are). For some simple cases like linear regression there's a closed-form formula, but for most real models (neural networks, logistic regression, etc.) there isn't - so we need an iterative search method, and gradient descent is the workhorse behind almost all of them. The core idea: the gradient of the loss function tells us the direction of steepest increase, so stepping in the opposite direction (negative gradient) decreases the loss; repeating this many times gradually walks the parameters toward a minimum. Understanding gradient descent by implementing it manually (rather than only calling `.fit()`) makes it much easier to reason about learning rate, convergence, and why training sometimes fails. The cell below just generates a simple linear dataset (y = 3x + 5 plus noise) that we'll fit by hand.

In [ ]:
import numpy as np

np.random.seed(0)
X = np.linspace(0, 10, 50)
y = 3 * X + 5 + np.random.randn(50) * 2  # true relationship is y = 3x + 5, with added Gaussian noise

## Implementing It
We're fitting a simple line y = m*x + b, so there are exactly two parameters to learn: slope `m` and intercept `b`. We start both at 0 (a reasonable, simple initial guess) and repeatedly: (1) compute predictions with the current m and b, (2) measure the error between predictions and actual y, (3) compute the gradient - how much the mean squared error would change if we nudged m or b slightly - and (4) update m and b by taking a small step (scaled by `learning_rate`) opposite to the gradient. `grad_m` and `grad_b` here are the derivatives of the MSE loss with respect to m and b respectively, derived from calculus. Doing this for 1000 epochs (passes over the data) should converge to values very close to the true slope (3) and intercept (5) used to generate the data - that's the "learned" output printed at the end.

In [ ]:
m, b = 0.0, 0.0  # initial guesses for slope and intercept
learning_rate = 0.01  # step size - too large can diverge, too small converges slowly
n = len(X)

for epoch in range(1000):  # repeatedly update m, b using the full dataset each time (batch gradient descent)
    y_pred = m * X + b
    error = y_pred - y
    grad_m = (2 / n) * np.dot(error, X)  # derivative of MSE loss w.r.t. m
    grad_b = (2 / n) * error.sum()  # derivative of MSE loss w.r.t. b
    m -= learning_rate * grad_m  # step opposite the gradient to reduce loss
    b -= learning_rate * grad_b

print('Learned slope:', m)
print('Learned intercept:', b)

## Tracking the Loss
Just checking the final learned m and b tells us the algorithm reached a good answer, but not how it got there. Recording the loss (mean squared error) at every epoch and plotting it lets us visually confirm that gradient descent is actually converging - the loss curve should drop quickly at first and then flatten out as m and b approach their optimal values. This is one of the most common diagnostic plots in ML: if the loss doesn't decrease, or bounces around erratically, it's usually a sign the learning rate is too high; if it decreases extremely slowly, the learning rate may be too low. Here we rerun gradient descent from scratch (resetting m and b to 0) for 200 epochs, this time storing the loss at each step in the `losses` list.

In [ ]:
import matplotlib.pyplot as plt

m, b = 0.0, 0.0  # reset parameters to rerun from scratch, this time tracking loss
losses = []
for epoch in range(200):
    y_pred = m * X + b
    error = y_pred - y
    loss = (error ** 2).mean()  # mean squared error - the quantity gradient descent is minimizing
    losses.append(loss)
    grad_m = (2 / n) * np.dot(error, X)
    grad_b = (2 / n) * error.sum()
    m -= learning_rate * grad_m
    b -= learning_rate * grad_b

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Gradient Descent Convergence')
plt.show()

## Comparing with scikit-learn
This is the sanity check: scikit-learn's `LinearRegression` doesn't use gradient descent internally (it solves the linear regression problem directly via a closed-form/least-squares solution), but it should arrive at essentially the same slope and intercept as our manual gradient descent implementation, since both are minimizing the same mean squared error objective. If the numbers from this cell closely match the "Learned slope"/"Learned intercept" from the from-scratch version above, that confirms our implementation is correct. Small differences are expected and fine - they just reflect that gradient descent is an approximate, iterative method that hasn't fully converged to the exact optimum, while sklearn's solver finds it exactly.

In [ ]:
from sklearn.linear_model import LinearRegression

sk_model = LinearRegression().fit(X.reshape(-1, 1), y)  # sklearn expects a 2D array of features, hence reshape
print('sklearn slope:', sk_model.coef_[0])
print('sklearn intercept:', sk_model.intercept_)